# Предшественники ФП/СФП за 365 дней: 2 подхода

Тетрадка считает TOP-5 предшественников ФП/СФП по сегментам (`МкБ`, `МСБ`, `ККБ`) для двух наборов целевых факторов:

1. `lines_1_3_only` — только 1-я и 3-я строки в сегменте (без строк в кавычках).
2. `all_lines` — все строки в сегменте (включая строки в кавычках).

Окно предшественников: **365 дней до даты целевого выявления** (строго раньше даты целевого события).

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

LOOKBACK_DAYS = 365
TOP_N = 5

# БАНКОВЫЕ пути (как в референсе analysis_default_logic_eda.ipynb).
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

# Период, в котором ищем целевые события.
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}

# Для сопоставления наименований из справочника.
SEGMENT_TO_ZO = {
    "МкБ": {"ДМ (микро)"},
    "МСБ": {"ДМСБ (малый)", "ДМСБ (средний)"},
    "ККБ": {"ДКБ"},
}

# Наборы целевых факторов по запросу.
TARGETS = {
    "lines_1_3_only": {
        "МкБ": {
            "ФП": ["13", "13У", "15.1.2"],
            "СФП": ["15.2У", "15.2.1У", "15.2"],
        },
        "МСБ": {
            "ФП": ["13", "15.1.3"],
            "СФП": ["15.2"],
        },
        "ККБ": {
            "ФП": ["13", "13.1", "15.1.3.1"],
            "СФП": ["15.2.1", "15.2.2", "15.2.1.1", "15.2.1.2"],
        },
    },
    "all_lines": {
        "МкБ": {
            "ФП": ["13", "13У", "15.1У", "15.1.1У", "15.1.1", "15.1.2"],
            "СФП": ["15.2У", "15.2.1У", "15.2"],
        },
        "МСБ": {
            "ФП": ["13", "15.1.2", "15.1.3"],
            "СФП": ["15.2"],
        },
        "ККБ": {
            "ФП": ["13", "13.1", "15.1.2 (II)", "15.1.2 (III)", "15.1.2.1", "15.1.3.1"],
            "СФП": ["15.2.1", "15.2.2", "15.2.1.1", "15.2.1.2"],
        },
    },
}


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def resolve_factor_name(segment, factor_type, factor_code):
    # Строгий exact-матч кода (без каких-либо вариантов/префиксов).
    return name_map.get((segment, factor_type, factor_code), "")


print("Конфигурация загружена.")
print(f"CRM_FILE: {CRM_FILE}")
print(f"REF_FILE: {REF_FILE}")
print(f"Период целей: {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}")
print(f"Окно предшественников: {LOOKBACK_DAYS} дней")

In [ ]:
crm_path = Path(CRM_FILE)
ref_path = Path(REF_FILE)

print("CRM:", crm_path)
print("REF:", ref_path)

# ── Загрузка CRM (1-в-1 как в референсе) ──
df = pd.read_csv(crm_path, dtype=str, low_memory=False)
df.columns = df.columns.str.strip()
print(f"Загружено: {len(df):,} строк × {df.shape[1]} колонок")

# ── Фильтр по источникам ──
if "VAL" in df.columns:
    before_src = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"Фильтр по источникам ({', '.join(ALLOWED_SOURCES)}): {len(df):,} строк (было {before_src:,})")
else:
    print("⚠ Колонка 'VAL' не найдена — фильтр по источникам не применён")

# ── Нормализация ИНН ──
df["inn"] = df["X_INN"].apply(normalize_inn)

# ── Преобразование даты ──
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

# ── Фильтр по периоду ──
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(df):,} строк")

# ── Нормализация TYPE_FP (строго как в референсе) ──
df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})

# ── Нормализация номера фактора (строго raw) ──
df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

# ── Маппинг сегментов ──
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print("\nНемаппированные X_AREA_RESP (исключены):")
    print(unmapped.to_string())

df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

# ── Удаление строк без номера фактора / ИНН ──
before_drop = len(df)
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(df):,} строк (удалено {before_drop - len(df):,})")

# ── Дедупликация (строго как в референсе) ──
before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации (INN + FP + TYPE + DATE): {len(df):,} (удалено {before_dedup - len(df):,})")
print(f"Контрольная метрика rows_after_dedup: {len(df):,}")

print(f"\nУникальных ИНН: {df['inn'].nunique():,}")
print(f"Уникальных номеров факторов: {df['fp_num'].nunique():,}")
print("\nРаспределение по сегментам:")
for seg in ["МкБ", "МСБ", "ККБ"]:
    n = df[df["segment"] == seg]["inn"].nunique()
    print(f"  {seg}: {n:,} ИНН")

# Рабочая база для расчета предшественников.
base = df.copy()


def load_ref(ref_file: Path):
    candidates = [
        {"sep": "\t", "encoding": "utf-16"},
        {"sep": ";"},
        {"sep": ","},
    ]
    for params in candidates:
        try:
            ref_df = pd.read_csv(ref_file, dtype=str, low_memory=False, **params)
            if len(ref_df.columns) > 1:
                return ref_df
        except Exception:
            continue
    return None


name_map = {}
ref = load_ref(ref_path)
if ref is None:
    raise FileNotFoundError(f"Не удалось прочитать ref_book.csv по пути: {ref_path}")

ref.columns = ref.columns.str.strip()
needed = {"№", "Наименование", "ЗО для ФП/СФП", "ФП", "СФП"}
if not needed.issubset(set(ref.columns)):
    raise KeyError("В ref_book.csv не хватает колонок: №, Наименование, ЗО для ФП/СФП, ФП, СФП")

ref = ref.copy()
ref["code"] = ref["№"].astype(str).str.strip()
ref["zo"] = ref["ЗО для ФП/СФП"].astype(str).str.strip()
ref["name"] = ref["Наименование"].astype(str).str.strip()

fp_rows = ref[ref["ФП"].astype(str).str.upper().eq("Y")][["code", "zo", "name"]].copy()
fp_rows["type"] = "ФП"
sfp_rows = ref[ref["СФП"].astype(str).str.upper().eq("Y")][["code", "zo", "name"]].copy()
sfp_rows["type"] = "СФП"
ref_long = pd.concat([fp_rows, sfp_rows], ignore_index=True)

temp = {}
for row in ref_long.itertuples(index=False):
    for seg, zo_set in SEGMENT_TO_ZO.items():
        if row.zo in zo_set:
            key = (seg, row.type, row.code)
            temp.setdefault(key, set()).add(row.name)

name_map = {k: " / ".join(sorted(v)) for k, v in temp.items()}
print(f"Сопоставление наименований собрано: {len(name_map):,} ключей")

In [ ]:
def build_target_table(target_cfg):
    rows = []
    for seg, type_dict in target_cfg.items():
        for target_type, code_list in type_dict.items():
            for code in code_list:
                rows.append({
                    "segment": seg,
                    "target_type": target_type,
                    "target_code": str(code).strip(),
                })
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


def compute_predecessors(base_df, target_cfg, approach_name):
    target_tbl = build_target_table(target_cfg)
    target_keys = set(zip(target_tbl["segment"], target_tbl["target_type"], target_tbl["target_code"]))

    work = base_df.copy()
    work["event_key"] = list(zip(work["segment"], work["TYPE_FP"], work["fp_num"]))

    target_events = work[
        work["dt"].between(DATE_FROM, DATE_TO)
        & work["event_key"].isin(target_keys)
    ][["inn", "segment", "dt", "TYPE_FP", "fp_num"]].copy()
    target_events = target_events.rename(columns={"dt": "target_dt", "TYPE_FP": "target_type", "fp_num": "target_code"})
    target_events = target_events.reset_index(drop=True)
    target_events["target_event_id"] = target_events.index.astype(int)

    target_coverage = (
        target_tbl.merge(
            target_events.groupby(["segment", "target_type", "target_code"], as_index=False)
            .size()
            .rename(columns={"size": "events_found"}),
            on=["segment", "target_type", "target_code"],
            how="left",
        )
        .fillna({"events_found": 0})
    )
    target_coverage["events_found"] = target_coverage["events_found"].astype(int)
    target_coverage.insert(0, "approach", approach_name)

    if len(target_events) == 0:
        empty_top = pd.DataFrame(
            columns=["approach", "segment", "rank_in_segment", "pred_type", "pred_code", "pred_name", "target_cases", "predecessor_events", "unique_inn"]
        )
        empty_summary = pd.DataFrame(
            [{"approach": approach_name, "segment": s, "target_events": 0, "target_inn": 0, "target_events_with_predecessor": 0} for s in ["МкБ", "МСБ", "ККБ"]]
        )
        return {
            "target_events": target_events,
            "target_coverage": target_coverage,
            "top5": empty_top,
            "summary": empty_summary,
        }

    pred_pool = work[["inn", "segment", "dt", "TYPE_FP", "fp_num"]].copy()

    merged = (
        target_events[["target_event_id", "inn", "segment", "target_dt", "target_type", "target_code"]]
        .merge(pred_pool, on=["inn", "segment"], how="left", suffixes=("", "_pred"))
        .rename(columns={"dt": "pred_dt", "TYPE_FP": "pred_type", "fp_num": "pred_code"})
    )

    merged = merged[
        (merged["pred_dt"] < merged["target_dt"])
        & (merged["pred_dt"] >= (merged["target_dt"] - pd.Timedelta(days=LOOKBACK_DAYS)))
    ].copy()

    merged_unique = merged.drop_duplicates(subset=["target_event_id", "pred_type", "pred_code"]).copy()

    if len(merged_unique) == 0:
        top5 = pd.DataFrame(
            columns=["approach", "segment", "rank_in_segment", "pred_type", "pred_code", "pred_name", "target_cases", "predecessor_events", "unique_inn"]
        )
    else:
        agg = (
            merged_unique.groupby(["segment", "pred_type", "pred_code"], as_index=False)
            .agg(
                target_cases=("target_event_id", "nunique"),
                predecessor_events=("pred_dt", "count"),
                unique_inn=("inn", "nunique"),
            )
            .sort_values(
                ["segment", "target_cases", "predecessor_events", "unique_inn", "pred_type", "pred_code"],
                ascending=[True, False, False, False, True, True],
            )
        )
        agg["rank_in_segment"] = agg.groupby("segment").cumcount() + 1
        top5 = agg[agg["rank_in_segment"] <= TOP_N].copy()

    top5.insert(0, "approach", approach_name)
    if len(top5):
        top5["pred_name"] = top5.apply(
            lambda r: resolve_factor_name(r["segment"], r["pred_type"], r["pred_code"]),
            axis=1,
        )
    else:
        top5["pred_name"] = ""

    summary = (
        target_events.groupby("segment", as_index=False)
        .agg(target_events=("target_event_id", "count"), target_inn=("inn", "nunique"))
    )
    if len(merged_unique) > 0:
        with_pred = (
            merged_unique.groupby("segment", as_index=False)
            .agg(target_events_with_predecessor=("target_event_id", "nunique"))
        )
        summary = summary.merge(with_pred, on="segment", how="left")
    else:
        summary["target_events_with_predecessor"] = 0
    summary["target_events_with_predecessor"] = summary["target_events_with_predecessor"].fillna(0).astype(int)

    for seg in ["МкБ", "МСБ", "ККБ"]:
        if seg not in set(summary["segment"]):
            summary = pd.concat(
                [summary, pd.DataFrame([{"segment": seg, "target_events": 0, "target_inn": 0, "target_events_with_predecessor": 0}])],
                ignore_index=True,
            )

    summary = summary.sort_values("segment")
    summary.insert(0, "approach", approach_name)

    return {
        "target_events": target_events,
        "target_coverage": target_coverage,
        "top5": top5,
        "summary": summary,
    }


results = {name: compute_predecessors(base, cfg, name) for name, cfg in TARGETS.items()}

summary_all = pd.concat([results[name]["summary"] for name in TARGETS], ignore_index=True)
coverage_all = pd.concat([results[name]["target_coverage"] for name in TARGETS], ignore_index=True)
top5_all = pd.concat([results[name]["top5"] for name in TARGETS], ignore_index=True)
target_events_all = pd.concat(
    [results[name]["target_events"].assign(approach=name) for name in TARGETS],
    ignore_index=True,
)

coverage_all["is_found"] = coverage_all["events_found"] > 0
missing_targets = coverage_all[~coverage_all["is_found"]].copy()
found_targets = coverage_all[coverage_all["is_found"]].copy()

print("Сводка по целевым событиям:")
display(summary_all.sort_values(["approach", "segment"]))

print("Покрытие целевых кодов (сколько событий найдено для каждого кода):")
display(coverage_all.sort_values(["approach", "segment", "target_type", "target_code"]))

print("\nДиагностика покрытия кодов:")
print(f"  Найдено кодов с событиями: {len(found_targets):,}")
print(f"  Не найдено кодов в периоде: {len(missing_targets):,}")
if len(missing_targets):
    display(
        missing_targets[["approach", "segment", "target_type", "target_code", "events_found"]]
        .sort_values(["approach", "segment", "target_type", "target_code"])
    )

for approach in TARGETS:
    print("=" * 95)
    print(f"TOP-{TOP_N} предшественников | подход: {approach}")
    print("=" * 95)
    cur = top5_all[top5_all["approach"] == approach].copy()
    if len(cur) == 0:
        print("Нет найденных предшественников для выбранных целей.")
        continue
    for seg in ["МкБ", "МСБ", "ККБ"]:
        print(f"\nСегмент: {seg}")
        sub = cur[cur["segment"] == seg].copy().sort_values("rank_in_segment")
        if len(sub) == 0:
            print("  Нет данных")
        else:
            display(
                sub[[
                    "rank_in_segment",
                    "pred_type",
                    "pred_code",
                    "pred_name",
                    "target_cases",
                    "predecessor_events",
                    "unique_inn",
                ]]
            )

In [ ]:
OUT_XLSX = Path(DATA_DIR) / "predecessors_top5_two_approaches.xlsx"

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    summary_all.sort_values(["approach", "segment"]).to_excel(writer, sheet_name="summary", index=False)
    coverage_all.sort_values(["approach", "segment", "target_type", "target_code"]).to_excel(writer, sheet_name="target_coverage", index=False)
    top5_all.sort_values(["approach", "segment", "rank_in_segment"]).to_excel(writer, sheet_name="top5_predecessors", index=False)
    target_events_all.sort_values(["approach", "segment", "target_dt"]).to_excel(writer, sheet_name="target_events", index=False)
    missing_targets.sort_values(["approach", "segment", "target_type", "target_code"]).to_excel(writer, sheet_name="missing_targets", index=False)

print(f"Сохранено: {OUT_XLSX}")